In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies before importing pandas/numpy in later cells.
!pip install -q pymysql sentence-transformers


Mounted at /content/drive


In [3]:
%%bash
cp /content/drive/MyDrive/TAWOS.sql /content/TAWOS.sql
ls -lh /content/TAWOS.sql
tail -c 300 /content/TAWOS.sql

-rw------- 1 root root 4.1G Oct 25 09:16 /content/TAWOS.sql
OLD_UNIQUE_CHECKS */;
/*!40101 SET CHARACTER_SET_CLIENT=@OLD_CHARACTER_SET_CLIENT */;
/*!40101 SET CHARACTER_SET_RESULTS=@OLD_CHARACTER_SET_RESULTS */;
/*!40101 SET COLLATION_CONNECTION=@OLD_COLLATION_CONNECTION */;
/*!40111 SET SQL_NOTES=@OLD_SQL_NOTES */;

-- Dump completed on 2022-09-27 16:12:14


In [4]:
%%bash
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq mariadb-server pv > /dev/null
service mariadb start
sleep 5
mysql -e "SELECT VERSION(); CREATE DATABASE IF NOT EXISTS tawos_cleaned CHARACTER SET utf8mb4;"

 * Starting MariaDB database server mariadbd
   ...done.
VERSION()
10.11.14-MariaDB-0ubuntu0.24.04.1


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
%%bash
set -o pipefail

# Import synchronously: this cell must finish before the export cells run.
# The compatibility substitutions adapt common MySQL 8 collations to MariaDB.
pv -f /content/TAWOS.sql \
 | sed -e 's/utf8mb4_0900_ai_ci/utf8mb4_general_ci/g' \
       -e 's/utf8mb4_0900_as_cs/utf8mb4_bin/g' \
       -e 's/utf8mb4_0900_bin/utf8mb4_bin/g' \
       -e '/GTID_PURGED/d' -e '/SQL_LOG_BIN/d' \
 | mysql --force --max_allowed_packet=1G tawos_cleaned \
   2> /content/import.log

echo "Import command finished. Review /content/import.log in the verification cell."


Import command finished. Review /content/import.log in the verification cell.


4.02GiB 0:06:48 [10.1MiB/s] [================================>] 100%            


In [6]:
import pymysql, pymysql.cursors, csv

def connect(stream=False):
    return pymysql.connect(unix_socket="/var/run/mysqld/mysqld.sock",
                           user="root", database="tawos_cleaned", charset="utf8mb4",
                           cursorclass=pymysql.cursors.SSCursor if stream else pymysql.cursors.Cursor)

issue_cols = ["ID", "Issue_Key", "Title", "Description_Text", "Type", "Priority",
              "Status", "Resolution", "Creation_Date", "Estimation_Date",
              "Resolution_Date", "Last_Updated", "Story_Point", "Project_ID",
              "Sprint_ID", "Creator_ID", "Reporter_ID", "Assignee_ID",
              "Title_Changed_After_Estimation",
              "Description_Changed_After_Estimation",
              "Story_Point_Changed_After_Estimation"]

def export(table, cols, path):
    n = 0
    conn = connect(stream=True)
    with conn.cursor() as cur, open(path, "w", newline="", encoding="utf-8") as f:
        cur.execute(f"SELECT {', '.join(cols) if cols else '*'} FROM {table}")
        w = csv.writer(f)
        w.writerow(cols if cols else [d[0] for d in cur.description])
        for row in cur:
            w.writerow(row); n += 1
    conn.close()
    return n

n_issue = export("Issue", issue_cols, "/content/issue.csv")
n_project = export("Project", None, "/content/project.csv")
print("Issue rows written:", n_issue, "| Project rows written:", n_project)

Issue rows written: 458232 | Project rows written: 39


In [7]:
import subprocess
import pandas as pd

# Confirm that the SQL import completed cleanly and the CSV export is complete.
count_result = subprocess.run(
    ["mysql", "-N", "-e", "SELECT COUNT(*) FROM Issue", "tawos_cleaned"],
    capture_output=True, text=True, check=True,
)
db_issue_count = int(count_result.stdout.strip())
export_df = pd.read_csv("/content/issue.csv", low_memory=False)
with open("/content/import.log", encoding="utf-8", errors="ignore") as log_file:
    import_log = log_file.read()
error_lines = [line for line in import_log.splitlines() if "ERROR" in line.upper()]

print(
    "MySQL Issue rows:", db_issue_count,
    "| exported rows:", n_issue,
    "| CSV rows:", len(export_df),
    "| unique IDs:", export_df["ID"].is_unique,
    "| import ERROR lines:", len(error_lines),
)
print("Expected rows from the previously checked dump: 458232")
assert db_issue_count == n_issue == len(export_df), "Database/export/CSV row counts do not match."
assert export_df["ID"].is_unique, "Duplicate issue IDs found in exported CSV."
assert not error_lines, f"SQL import reported errors; inspect /content/import.log. First errors: {error_lines[:5]}"


MySQL Issue rows: 458232 | exported rows: 458232 | CSV rows: 458232 | unique IDs: True | import ERROR lines: 0
Expected rows from the previously checked dump: 458232


In [8]:
!cp /content/issue.csv /content/project.csv /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/issue.csv

-rw------- 1 root root 366M Oct 25 09:24 /content/drive/MyDrive/issue.csv


## Preprocessing and label policy (Problem 6)

The project is **SBERT sentence embeddings, within-project estimation, chronological/temporal split**.

The preprocessing cell below applies the label rule immediately: retain non-missing story points in the provisional range **0 <= Story_Point <= 100 inclusive**. It intentionally keeps zero and fractional values. Values above 100 are excluded as provisional outliers; report this choice and its impact. If the instructor has specified a different label rule, change the cutoff here before running any downstream cells.

Text is built from the cleaned title plus description when available, and falls back to title-only when description is blank. Jira `{code}` and `{noformat}` contents are retained while their wrapper markers are removed. Issues with fewer than three words of final text are excluded.

In [9]:
import pandas as pd, re

INPUT_PATH = "/content/drive/MyDrive/issue.csv"
OUTPUT_PATH = "/content/drive/MyDrive/issue_clean.csv"

# Load the exported TAWOS issues. Keep original columns for auditability.
df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["Creation_Date", "Estimation_Date", "Resolution_Date", "Last_Updated"],
    low_memory=False,
)
print("raw rows:", len(df))

required_columns = {"ID", "Project_ID", "Title", "Description_Text",
                    "Story_Point", "Creation_Date"}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Missing required columns in issue.csv: {missing_columns}"

# Ensure story points are numeric before filtering; invalid non-numeric values become missing.
df["Story_Point"] = pd.to_numeric(df["Story_Point"], errors="coerce")

# 1. Keep labeled issues with non-negative story points up to the provisional
#    upper bound of 100. This intentionally preserves 0 and fractional labels.
sp = df.loc[
    df["Story_Point"].notna()
    & (df["Story_Point"] >= 0)
    & (df["Story_Point"] <= 100)
].copy()
print("labeled numeric rows:", int(df["Story_Point"].notna().sum()))
print("non-numeric/missing story-point rows excluded:", int(df["Story_Point"].isna().sum()))
print("after keeping 0 <= Story_Point <= 100:", len(sp))
print("zero labels kept:", int((sp["Story_Point"] == 0).sum()))
print("fractional labels kept:", int((sp["Story_Point"] % 1 != 0).sum()))
print("excluded labels above 100:", int((df["Story_Point"] > 100).sum()))

# 2. Normalize raw text carefully. Replace literal escaped line breaks/tabs.
def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = s.replace("\\r\\n", " ").replace("\\n", " ").replace("\\t", " ")
    s = s.replace("\r\n", " ").replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", s).strip()

def fix_quotes(s):
    s = re.sub(r'"{2,}', '"', s).strip()
    if len(s) > 1 and s.startswith('"') and s.endswith('"'):
        s = s[1:-1].strip()
    return s

def strip_markup(s):
    if not s:
        return ""

    # 1. Protect URLs before removing any markup.
    # Keep the original URL spelling and characters.
    protected_urls = []

    def protect_url(match):
        protected_urls.append(match.group(0))
        return f"ZZURLTOKEN{len(protected_urls) - 1}ZZ"

    # Capture URL-like tokens. Trim common surrounding markup punctuation
    # from the end of each match, without changing the original URL.
    def protect_url_token(match):
        token = match.group(0)
        url = token

        while url and url[-1] in ".,;":
            url = url[:-1]

        # Exclude closing markup delimiters when they trail a URL.
        while url and url[-1] in "]}":
            url = url[:-1]

        suffix = token[len(url):]
        if url.lower().startswith(("http://", "https://")):
            return protect_url(
                re.match(r".*", url, flags=re.S)
            ) + suffix
        return token

    s = re.sub(
        r'https?://[^\s<>"\']+',
        protect_url_token,
        s,
        flags=re.I
    )

    # 2. Protect code/noformat blocks from markup cleanup.
    protected_blocks = []

    def protect_block(match):
        protected_blocks.append(match.group(1))
        return f"ZZPROTECTEDBLOCK{len(protected_blocks) - 1}ZZ"

    s = re.sub(
        r"\{code[^}]*\}(.*?)\{code\}",
        protect_block,
        s,
        flags=re.S | re.I
    )
    s = re.sub(
        r"\{noformat\}(.*?)\{noformat\}",
        protect_block,
        s,
        flags=re.S | re.I
    )

    # 3. Convert Jira links to readable label + URL.
    s = re.sub(
        r"\[([^\]|]+)\|([^\]]+)\]",
        r"\1 (\2)",
        s
    )

    # 4. Remove common Jira formatting/control markers.
    s = re.sub(r"\{[A-Za-z]+(?::[^}]*)?\}", " ", s)
    s = re.sub(r"(?<!\w)\*(.*?)\*(?!\w)", r"\1", s)

    # 5. Restore code/noformat block contents.
    for i, block in enumerate(protected_blocks):
        s = s.replace(f"ZZPROTECTEDBLOCK{i}ZZ", block)

    # 6. Remove the dataset placeholder.
    s = s.replace("<USER>", "")

    # 7. Restore protected URLs.
    for i, url in enumerate(protected_urls):
        s = s.replace(f"ZZURLTOKEN{i}ZZ", url)

    return re.sub(r"\s+", " ", s).strip()

def full_clean(s):
    return strip_markup(fix_quotes(clean_text(s)))

# Preserve the title + original description before markup/URL cleaning for auditing.
# This column is NOT fed to SBERT; SBERT uses the cleaned `text` column below.
sp["Title_raw"] = sp["Title"].fillna("").astype(str).str.strip()
sp["Desc_raw"] = sp["Description_Text"].fillna("").astype(str).str.strip()
sp["text_raw"] = (
    sp["Title_raw"] + " " + sp["Desc_raw"]
).str.replace(r"\s+", " ", regex=True).str.strip()

sp["Title_clean"] = sp["Title"].map(full_clean)
sp["Desc_clean"] = sp["Description_Text"].map(full_clean)

# 3. Use title + cleaned description when available; otherwise use cleaned title.
#    This also handles descriptions that are NaN or become empty after cleaning.
sp["text"] = sp["Title_clean"]
has_description = sp["Desc_clean"].ne("")
sp.loc[has_description, "text"] = (
    sp.loc[has_description, "Title_clean"] + " " + sp.loc[has_description, "Desc_clean"]
).str.strip()
sp["text"] = sp["text"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
print("rows still containing <USER>:", int(sp["text"].str.contains("<USER>", regex=False).sum()))
assert not sp["text"].str.contains("<USER>", regex=False).any(), "<USER> placeholder remains in cleaned text."

# 4. Drop only rows with unusable/very short final text. Do not modify labels.
short_mask = sp["text"].str.split().str.len() < 3
print("rows with fewer than 3 words in final text:", int(short_mask.sum()))
sp = sp.loc[~short_mask].copy()

# Remove the unused constant/change-tracking column as in the original notebook.
if "Description_Changed_After_Estimation" in sp.columns:
    sp = sp.drop(columns=["Description_Changed_After_Estimation"])

sp = sp.sort_values(["Creation_Date", "ID"]).reset_index(drop=True)
print("final rows:", len(sp))
print("projects:", sp["Project_ID"].nunique())
print("zero labels in final data:", int((sp["Story_Point"] == 0).sum()))
print("fractional labels in final data:", int((sp["Story_Point"] % 1 != 0).sum()))
print("empty final texts:", int(sp["text"].eq("").sum()))
print("exact duplicate texts:", int(sp["text"].duplicated().sum()))
print("sample final texts:")
print(sp[["Title", "Description_Text", "text_raw", "text", "Story_Point"]].head(3).to_string(index=False))

sp.to_csv(OUTPUT_PATH, index=False)
print("Saved cleaned dataset to:", OUTPUT_PATH)


raw rows: 458232
labeled numeric rows: 65427
non-numeric/missing story-point rows excluded: 392805
after keeping 0 <= Story_Point <= 100: 65286
zero labels kept: 2416
fractional labels kept: 3328
excluded labels above 100: 141
rows still containing <USER>: 0
rows with fewer than 3 words in final text: 271
final rows: 65015
projects: 39
zero labels in final data: 2408
fractional labels in final data: 3311
empty final texts: 0
exact duplicate texts: 603
sample final texts:
                                   Title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [10]:
from pathlib import Path
import pandas as pd

CLEAN_PATH = "/content/drive/MyDrive/issue_clean.csv"
PROJECT_PATH = "/content/drive/MyDrive/project.csv"
assert Path(CLEAN_PATH).exists(), f"Cleaned file not found: {CLEAN_PATH}. Run preprocessing cell first."
assert Path(PROJECT_PATH).exists(), f"Project file not found: {PROJECT_PATH}. Run the export cell first."

sp = pd.read_csv(CLEAN_PATH, parse_dates=["Creation_Date"], low_memory=False)
proj = pd.read_csv(PROJECT_PATH, low_memory=False)

required = {"ID", "Project_ID", "Story_Point", "Creation_Date", "text"}
missing = required - set(sp.columns)
assert not missing, f"Missing columns in cleaned issues: {missing}"
assert sp["ID"].is_unique, "Issue ID values must be unique."
assert sp["text"].notna().all() and sp["text"].str.strip().ne("").all(), "Found empty final text."

print("Cleaned issues shape:", sp.shape)
print("Projects file shape:", proj.shape)
print("Project columns:", proj.columns.tolist())
print("First 3 project rows:")
print(proj.head(3).to_string(index=False))

counts = (sp.groupby("Project_ID").size().rename("n_issues").sort_values(ascending=False))
print("\nIssues per project:")
print(counts.to_string())
print("\nUnique projects in cleaned issues:", sp["Project_ID"].nunique())
print("Zero labels:", int((sp["Story_Point"] == 0).sum()))
print("Fractional labels:", int((sp["Story_Point"] % 1 != 0).sum()))

Cleaned issues shape: (65015, 26)
Projects file shape: (39, 9)
Project columns: ['ID', 'Project_Key', 'Name', 'URL', 'Description', 'Start_Date', 'Last_Update_Date', 'SP_Field_ID', 'Repository_ID']
First 3 project rows:
 ID Project_Key            Name                                         URL                                                                                                                                                 Description          Start_Date    Last_Update_Date       SP_Field_ID  Repository_ID
  1          XD       Spring XD         https://jira.spring.io/projects/XD/          Spring XD makes it easy to solve common big data problems such as data ingestion and export, real-time analytics, and batch workflow orchestration 2013-04-12 06:37:17 2018-11-20 15:18:11 customfield_10142              1
  2    DATACASS Spring DataCass   https://jira.spring.io/projects/DATACASS/ The Apache Cassandra NoSQL Database offers many new capabilities for teams seeking a solution 

In [11]:
# Summarize each project and attach its human-readable name/key.
# The export's Project table is expected to contain ID, Project_Key, and Name.
required_project_cols = {"ID", "Project_Key", "Name"}
missing_project_cols = required_project_cols - set(proj.columns)
assert not missing_project_cols, f"Missing columns in project.csv: {missing_project_cols}"

names = proj[["ID", "Project_Key", "Name"]].rename(columns={"ID": "Project_ID"})
summary = (
    sp.groupby("Project_ID")
      .agg(
          n_issues=("ID", "size"),
          first=("Creation_Date", "min"),
          last=("Creation_Date", "max"),
          median_sp=("Story_Point", "median"),
          max_sp=("Story_Point", "max"),
      )
      .reset_index()
      .merge(names, on="Project_ID", how="left", validate="one_to_one")
      .sort_values("n_issues", ascending=False)
)
print(summary.to_string(index=False))
summary.to_csv("/content/drive/MyDrive/project_summary.csv", index=False)
print("Saved /content/drive/MyDrive/project_summary.csv")

 Project_ID  n_issues               first                last  median_sp  max_sp Project_Key                                Name
         28     20378 2014-03-04 11:32:23 2020-10-22 14:51:07        3.0   100.0          DM            Lsstcorp Data management
         43      5390 2014-06-03 22:41:30 2020-10-21 22:09:03        2.0     8.0         EVG             The MongoDB Engineering
         12      4655 2011-03-29 16:48:47 2020-10-17 15:41:42        5.0    63.0       TIMOB                   The Titanium SDK 
         36      4170 2009-02-28 00:59:32 2020-04-17 14:31:53        4.0    60.0        MULE                                Mule
          1      3695 2013-04-12 06:37:17 2018-11-16 15:57:31        3.0   100.0          XD                           Spring XD
         13      3405 2011-03-01 11:50:44 2020-05-12 02:15:14        5.0    55.0      TISTUD                 Appcelerator Studio
          4      3262 2012-11-19 21:12:23 2020-08-12 10:24:47        3.0    40.0       MESOS     

In [12]:
import json, math
import numpy as np
import pandas as pd

CLEAN_PATH = "/content/drive/MyDrive/issue_clean.csv"
SPLIT_PATH = "/content/drive/MyDrive/splits_temporal.json"
sp = pd.read_csv(CLEAN_PATH, parse_dates=["Creation_Date"], low_memory=False)

# Chronological evaluation requires a valid creation date and unique issue IDs.
sp = sp.dropna(subset=["Creation_Date"]).copy()
sp["ID"] = pd.to_numeric(sp["ID"], errors="raise").astype("int64")
sp["Project_ID"] = pd.to_numeric(sp["Project_ID"], errors="raise").astype("int64")
assert sp["ID"].is_unique, "Issue IDs must be unique before splitting."

MIN_ISSUES = 100
N_FOLDS = 5  # five chronological validation folds within the 80% development set
splits = {}
excluded_small = []

for pid, d in sp.groupby("Project_ID"):
    d = d.sort_values(["Creation_Date", "ID"], kind="stable")
    ids = d["ID"].astype(int).tolist()
    if len(ids) < MIN_ISSUES:
        excluded_small.append(int(pid))
        continue

    n_test = max(1, math.ceil(0.20 * len(ids)))
    dev, test = ids[:-n_test], ids[-n_test:]
    # Split development chronologically into 6 blocks: block 0 seeds training;
    # each of the following 5 blocks serves once as a future validation block.
    blocks = [list(map(int, b)) for b in np.array_split(dev, N_FOLDS + 1)]
    folds = []
    for i in range(1, N_FOLDS + 1):
        train_ids = [issue_id for block in blocks[:i] for issue_id in block]
        val_ids = blocks[i]
        if train_ids and val_ids:
            folds.append({"train": train_ids, "val": val_ids})
    if len(folds) != N_FOLDS:
        print(f"Skipping project {pid}: could not create {N_FOLDS} non-empty folds.")
        continue
    splits[int(pid)] = {"dev": dev, "test": test, "folds": folds}

with open(SPLIT_PATH, "w", encoding="utf-8") as f:
    json.dump(splits, f)
print("Saved splits:", SPLIT_PATH)
print("Projects included:", len(splits))
print("Validation method: 5-fold rolling-origin temporal validation (training uses only earlier blocks).")
print("Projects excluded for fewer than", MIN_ISSUES, "issues:", excluded_small)

# Sanity checks: all IDs exist; each train/validation/test period is chronological.
created = sp.set_index("ID")["Creation_Date"]
issues = []
for pid, split in splits.items():
    if set(split["dev"]) & set(split["test"]):
        issues.append((pid, "dev/test overlap"))
    if created.loc[split["dev"]].max() > created.loc[split["test"]].min():
        issues.append((pid, "dev/test chronology"))
    for fold_num, fold in enumerate(split["folds"], start=1):
        if set(fold["train"]) & set(fold["val"]):
            issues.append((pid, f"fold {fold_num} overlap"))
        if created.loc[fold["train"]].max() > created.loc[fold["val"]].min():
            issues.append((pid, f"fold {fold_num} chronology"))
print("Temporal split violations:", len(issues))
if issues:
    print(issues[:20])
assert not issues, "Temporal split sanity checks failed."
assert len(splits) > 0, "No projects qualified for temporal evaluation."

Saved splits: /content/drive/MyDrive/splits_temporal.json
Projects included: 39
Validation method: 5-fold rolling-origin temporal validation (training uses only earlier blocks).
Projects excluded for fewer than 100 issues: []
Temporal split violations: 0


In [13]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import importlib.util
import subprocess
import sys
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

CLEAN_PATH = "/content/drive/MyDrive/issue_clean.csv"
EMB_PATH = "/content/drive/MyDrive/sbert_minilm_emb.npy"
IDS_PATH = "/content/drive/MyDrive/sbert_ids.csv"
sp = pd.read_csv(CLEAN_PATH, low_memory=False)
sp["ID"] = pd.to_numeric(sp["ID"], errors="raise").astype("int64")
assert sp["ID"].is_unique, "Issue IDs must be unique."
assert sp["text"].notna().all() and sp["text"].str.strip().ne("").all(), "Empty text found."

# Reproducibility settings and automatic GPU/CPU selection.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
model.max_seq_length = 256
emb = model.encode(
    sp["text"].tolist(),
    batch_size=128 if device == "cuda" else 32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,
).astype("float32")

assert emb.shape == (len(sp), 384), f"Unexpected embedding shape: {emb.shape}"
assert np.isfinite(emb).all(), "Embeddings contain NaN or infinite values."
np.save(EMB_PATH, emb)
sp[["ID"]].to_csv(IDS_PATH, index=False)

# Reload saved artifacts and verify row counts and exact ID order.
emb_check = np.load(EMB_PATH, mmap_mode="r")
ids_check = pd.read_csv(IDS_PATH)
ids_check["ID"] = pd.to_numeric(ids_check["ID"], errors="raise").astype("int64")
assert len(emb_check) == len(sp) == len(ids_check), "Row count mismatch among text, embeddings, and IDs."
assert np.array_equal(ids_check["ID"].to_numpy(), sp["ID"].to_numpy()), "Saved ID order does not match embedding order."
print("Embedding generation complete.")
print("Embedding shape:", emb_check.shape, "| dtype:", emb_check.dtype)
print("NaNs:", int(np.isnan(emb_check).sum()), "| rows match:", len(emb_check) == len(sp) == len(ids_check))
print("Saved:", EMB_PATH, "and", IDS_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/508 [00:00<?, ?it/s]

Embedding generation complete.
Embedding shape: (65015, 384) | dtype: float32
NaNs: 0 | rows match: True
Saved: /content/drive/MyDrive/sbert_minilm_emb.npy and /content/drive/MyDrive/sbert_ids.csv


### Diagnostic check: URL preservation audit

This cell is a diagnostic audit of URL retention after preprocessing. It is not part of the embedding or model-training pipeline.


In [14]:
import re
import pandas as pd

# Match HTTP/HTTPS URLs in text
URL_PATTERN = re.compile(r'https?://[^\s<>"\'\]\)]+', re.IGNORECASE)

def extract_urls(text):
    if pd.isna(text):
        return []
    return URL_PATTERN.findall(str(text))

# Check that the required columns exist
assert {"text_raw", "text"}.issubset(sp.columns), (
    "Required columns 'text_raw' and 'text' are missing from sp."
)

total_original_urls = 0
total_preserved_urls = 0
total_missing_urls = 0
issues_with_urls = 0
missing_examples = []

for idx, row in sp.iterrows():
    original_urls = extract_urls(row["text_raw"])
    cleaned_text = str(row["text"]) if pd.notna(row["text"]) else ""

    if original_urls:
        issues_with_urls += 1

    total_original_urls += len(original_urls)

    for url in original_urls:
        # Compare case-insensitively; preserve the original URL spelling.
        if url.casefold() in cleaned_text.casefold():
            total_preserved_urls += 1
        else:
            total_missing_urls += 1

            if len(missing_examples) < 10:
                missing_examples.append({
                    "ID": row.get("ID", idx),
                    "original_url": url,
                    "text_raw": str(row["text_raw"])[:300],
                    "cleaned_text": cleaned_text[:300],
                })

print("URL preservation audit")
print("----------------------")
print("Issues checked:", len(sp))
print("Issues containing original URLs:", issues_with_urls)
print("Original URL occurrences:", total_original_urls)
print("URL occurrences found in cleaned text:", total_preserved_urls)
print("URL occurrences not found:", total_missing_urls)

if total_original_urls:
    print(
        "Preservation rate:",
        f"{100 * total_preserved_urls / total_original_urls:.2f}%"
    )
else:
    print("No HTTP/HTTPS URLs were detected in text_raw.")

if missing_examples:
    print("\nFirst missing URL examples:")
    display(pd.DataFrame(missing_examples))
else:
    print("\nAll detected original URL occurrences were found in cleaned text.")

URL preservation audit
----------------------
Issues checked: 65015
Issues containing original URLs: 11267
Original URL occurrences: 19232
URL occurrences found in cleaned text: 19227
URL occurrences not found: 5
Preservation rate: 99.97%

First missing URL examples:


,ID,original_url,text_raw,cleaned_text
0,183144,http://www.evernote.com|height=500},"""Default user macro parameters ignored"" """"""Use...",Default user macro parameters ignored User mac...
1,398292,http://IP:PORT,"""Pass in proxy details to Titanium CLI login c...",Pass in proxy details to Titanium CLI login ca...
2,407836,http://www.stephaniequinn.com/Music/Vivaldi%20...,"""iOS - Ti.Media.AudioPlayer - error after slee...",iOS - Ti.Media.AudioPlayer - error after sleep...
3,553,http://hadoop1.example.com:34549/modules?size=...,"""Spring XD shell crashes when network connecti...",Spring XD shell crashes when network connectio...
4,264621,https://cloud.google.com/compute/pricing#custo...,"""Support custom machine types in Google Comput...",Support custom machine types in Google Compute...


## End of notebook: SBERT embeddings

This notebook ends after creating SBERT sentence embeddings and saving the matching issue IDs. It does not include baseline evaluation or a story-point prediction model.

The `text_raw` column contains the combined original title and description before markup cleanup (whitespace is normalized). The cleaned `text` column is used for SBERT. Standalone HTTP/HTTPS URLs are preserved, and Jira `[label|URL]` links are converted to `label (URL)` so both the visible label and URL remain in the model input. Jira `{code}` and `{noformat}` wrappers are removed while their contents are protected from other markup-removal rules.

The temporal split file is created before embeddings for reproducibility. Validation uses rolling-origin chronological folds: each validation block occurs after its training data. Projects with fewer than 100 eligible issues are excluded from these splits; this is a project-specific design choice, not a requirement stated by the dataset itself.


**Reporting notes:** `text_raw` is an audit column only; it is not embedded, so this notebook does not perform a raw-versus-cleaned ablation. The first rolling-origin fold has a small initial training block (about 14 issues for a project with 100 eligible issues); report this limitation alongside results for small projects. Markup patterns not explicitly handled by `strip_markup` may remain or be removed imperfectly, so describe preprocessing limitations transparently. TAWOS URLs are partially anonymised, so some preserved URLs are incomplete. URL audit (diagnostic): 99.97% of URL occurrences in `text_raw` also appear in the cleaned text; the few exceptions occur inside Jira macros that are removed with the macro.
